In [1]:
# 0.加载系统环境变量
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# 1.准备工具
# 1.1 定义工具函数
def get_weather(location:str):
    """
    根据地点或坐标获取一个位置的天气信息
    :param location: 地点或者坐标，比如：北京
    :return: 当前位置的天气信息
    """
    return f"{location} 天气晴朗~"

# 1.2 组装工具描述信息
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "根据地点或坐标获取一个位置的天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "地点或者坐标，比如：北京",
                    },
                },
                "required": ["location"],
            },
        },
    },
]
# 1.3 将所有工具存放到字典中，方便第4步本地调用的时候查找到对应的工具函数
tool_dict = {
    "get_weather":get_weather
}

In [3]:
from openai import OpenAI

# 2.第一次调用 - 目的:让大模型推理分析要调用哪个工具
# 2.1 创建OpenAI客户端
client = OpenAI(
    # 注意:删除掉真正的系统环境变量中的配置
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
# 2.2 准备消息列表-调用大模型

# 注意:复数
messages = [
    {"role":"user", "content":"北京的天气怎么样?"}
]

response = client.chat.completions.create(
    model="deepseek-flash",
    messages=messages,
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    },
    tools=tools
)

message = response.choices[0].message
print(message.model_dump_json())

{"content":"","refusal":null,"role":"assistant","annotations":null,"audio":null,"function_call":null,"tool_calls":[{"id":"call_00_iOoAY4WuCWv7Fn4PEweg5232","function":{"arguments":"{\"location\": \"北京\"}","name":"get_weather"},"type":"function","index":0}]}


In [4]:
import json

# 3.解析返回结果
tool_calls = message.tool_calls[0]

# 3.1 取出调用ID
tool_call_id = tool_calls.id
# 3.2 取出要调用的函数名称
tool_name = tool_calls.function.name
# 3.3 取出要调用的函数参数
args = json.loads(tool_calls.function.arguments)

print(tool_call_id)
print(tool_name)
print(args)

call_00_iOoAY4WuCWv7Fn4PEweg5232
get_weather
{'location': '北京'}


In [5]:
# 4.本地工具函数调用
result = tool_dict[tool_name](**args)
print(result)

北京 天气晴朗~


In [6]:
# 5.第二次调用 - 目的:将工具的执行结果给到大模型，得到最终的答案

# 5.1 将AI的回答添加到消息列表中
messages.append(message)
# 5.2 将工具的执行结果添加到消息列表中
messages.append({
    "role":"tool",
    "content":result,
    "tool_call_id":tool_call_id
})
# 5.3 执行大模型调用
response = client.chat.completions.create(
    model="deepseek-flash",
    messages=messages,
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    },
    tools=tools
)

print(response.choices[0].message.content)

北京现在是晴天，天气晴朗。出门的话记得根据实时温度适当增减衣物，注意防晒～
